# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the FAIR² Mental Health Survey dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is defined by a Croissant schema URL:

- **Croissant schema URL:**
  [`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

The dataset contains survey responses from residents of Kilifi County, Kenya, including key demographic attributes and standardized mental health assessment scores (GAD-7, PHQ-9, PSQ, etc.). All data fields, record sets, and columns are referenced by their unique `@id`, ensuring reproducibility and clarity.

In [ ]:
# Ensure `mlcroissant` is installed (if not already)
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and explore the overall description using `mlcroissant`. The Croissant schema provides standardized metadata describing the structure and contents of the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: ", metadata.name)
print("Dataset Description: ", metadata.description)
print("Authors: ", metadata.author if hasattr(metadata, 'author') else None)
print("Keywords: ", metadata.keywords if hasattr(metadata, 'keywords') else None)


## 2. Data Overview

Let's review the available record sets, fields, and their `@id` values. This allows us to understand the structure and select entities for further exploration.

All entities are referenced by their unique `@id`.

In [ ]:
# List record sets and their fields (referencing by @id)
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []

if record_sets:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']} -- Name: {rs.get('name', 'N/A')}")
        fields = rs.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            print(f"\tField @id: {field['@id']} -- Name: {field.get('name', 'N/A')} -- DataType: {field.get('dataType', 'N/A')}")
else:
    print("No record sets found in metadata.")

# If record sets are not directly included, try loading records from a likely main record set
# Typical practice: many Croissant datasets have a main record set @id. 
# For demonstration, let's attempt with an example @id.
example_record_set_id = None
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print(f"\nSample records from record set {example_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i > 4:
            break

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s as listed in the overview.

For demonstration, we process all available record sets. If only one is present, we'll extract its contents.

In [ ]:
dataframes = {}
record_set_ids = []

if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    # Fallback: guess a main record set id
    record_set_ids = [example_record_set_id] if example_record_set_id else []

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set: {rs_id} -- Columns: {df.columns.tolist()}")
            print(df.head(3))
        else:
            print(f"No records found for record set: {rs_id}")
    except Exception as ex:
        print(f"Error loading records from {rs_id}: {ex}")

# Select a record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    df_main = dataframes[main_record_set_id]
    print(f"\nMain record set dataframe columns: {df_main.columns.tolist()}")
    df_main.head()
else:
    df_main = None

## 4. Exploratory Data Analysis (EDA)

We'll apply filtering, normalization, and grouping to the record set data. Use column `@id`s for referencing fields.

- Filter rows by a numeric field (e.g., GAD-7 score)
- Normalize scores
- Group results by a categorical field (e.g., level_of_education)

**Adjust the field `@id`s below to match your dataset structure as discovered above.**

In [ ]:
# Example EDA: filter, normalize, group.
if df_main is not None:
    # Find a numeric column by @id (e.g., GAD-7 score, PHQ-9, PSQ)
    possible_numeric_fields = [col for col in df_main.columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower() or 'score' in col.lower()]
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]
        print(f"Using numeric field @id: {numeric_field_id}")
        # Set a threshold for filtering
        threshold = 10
        # Filter records
        filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field (e.g., level_of_education)
        possible_group_fields = [col for col in df_main.columns if 'education' in col.lower() or 'level' in col.lower() or 'village' in col.lower()]
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field detected for filtering and normalization.")
else:
    print("No main dataframe available for EDA.")

## 5. Visualization

Visualize numeric field distributions and relationships with categorical attributes. For example, plot histogram of GAD-7 scores, or boxplots by education level.

For demonstration, the field `@id`s are dynamically determined from above.

In [ ]:
import matplotlib.pyplot as plt

if df_main is not None and possible_numeric_fields:
    plt.figure(figsize=(8, 5))
    df_main[numeric_field_id].dropna().hist(bins=20, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if possible_group_fields:
        group_field_id = possible_group_fields[0]
        plt.figure(figsize=(10, 6))
        df_main.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f"{numeric_field_id} distribution by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

We explored the FAIR² Mental Health Survey Dataset using `mlcroissant`, loading metadata and records directly by Croissant schema `@id`. By reviewing record sets, filtering and grouping by demographic attributes, and visualizing distributions and relationships, we've revealed basic insights into the surveyed mental health landscape of Kilifi County, Kenya.

- The use of standardized field and record set `@id`s ensures reproducibility and consistent referencing.
- Further analyses could include deeper statistical summaries, handling missing data, and advanced predictive modeling.
- This notebook can be extended to additional record sets and fields as needed.

_FAIR² dataset demonstrates AI-ready standards for community health science in Africa._

<!-- End of notebook -->